#  Livrable 1 – Classification binaire d’images

##  Objectif
Développer un modèle de **classification binaire** capable de distinguer :
- **Photos**
- **Autres images** (schémas, textes scannés, peintures…)

Ce livrable constitue la **première étape** du workflow demandé par TouNum.

### Lib import

In [ ]:
from keras.src.layers import MaxPooling2D, Conv2D, Dense
from keras import models, layers
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

#### Model parameters

In [ ]:
img_width = 150
img_height = 150
data_dir = "data/raw"
num_classes = 2
batch_size = 32
epochs = 100

In [ ]:
# separate data into training and testing sets
train_raw = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(img_width, img_height),
    batch_size=batch_size,
)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

train = train_raw.map(lambda x, y: (data_augmentation(x, training=True), y))

test_raw = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(img_width, img_height),
    batch_size=batch_size,
)

In [ ]:
# get class names
class_names = train_raw.class_names
print(class_names)

In [ ]:
# now apply .ignore_errors and the pipeline
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.ignore_errors().cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_raw.ignore_errors().cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# visualization (use class_names captured above)
plt.figure(figsize=(10, 10))
for images, labels in train_raw.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i].numpy())])
        plt.axis("off")

In [ ]:
# standardize data
normalization_layer = layers.Rescaling(1.0 / 255)
normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0].numpy()

In [ ]:
# verify the data is normalized
print(np.min(first_image), np.max(first_image))

In [ ]:
# Change output layer to 1 neuron with sigmoid
model = models.Sequential([
    layers.Input(shape=(img_with, img_height, 3)),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # Binary output
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

In [ ]:
# plot model summary
model.summary()

In [ ]:
# train model with EarlyStopping
from keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=1000,  # Large number for indefinite training
    callbacks=[early_stop]
 )

In [ ]:
# plot training and validation accuracy and loss
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
precision = history.history['precision']
val_precision = history.history['val_precision']
recall = history.history['recall']
val_recall = history.history['val_recall']
auc = history.history['auc']
val_auc = history.history['val_auc']

In [ ]:
# plot false positives and false negatives
plt.figure(figsize=(16, 8))
plt.subplot(2, 3, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

In [ ]:
# print images with predicted labels
plt.subplot(2, 3, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.subplot(2, 3, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.xlabel('Epoch')
plt.ylabel('Precision')
plt.legend(loc='lower right')
plt.title('Training and Validation Precision')
plt.subplot(2, 3, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.xlabel('Epoch')
plt.ylabel('Recall')
plt.legend(loc='lower right')
plt.title('Training and Validation Recall')
plt.subplot(2, 3, 5)
plt.plot(auc, label='Training AUC')
plt.plot(val_auc, label='Validation AUC')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend(loc='lower right')
plt.title('Training and Validation AUC')
plt.tight_layout()
plt.show()


In [ ]:
# load model and test on random images
model.save('liv1_LH_classification_model.h5')
from keras.models import load_model
model = load_model('liv1_LH_classification_model.h5')
import random
from keras.utils import load_img, img_to_array
for i in range(9):
    plt.subplot(3, 3, i + 1)
    folder = random.choice(os.listdir(data_dir))
    img_path = os.path.join(data_dir, folder, random.choice(os.listdir(os.path.join(data_dir, folder))))
    img = load_img(img_path, target_size=(img_with, img_height))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array)
    predicted_class = class_names[1] if prediction[0][0] > 0.5 else class_names[0]
    plt.imshow(img)
    plt.title(f'Predicted: {predicted_class}')
    plt.axis('off')
# import os
plt.tight_layout()
plt.show()